NEURAL FORECASTING MODEL

for affi


In [ ]:
"""
NeuroForcast — Transformer Encoder-Decoder with Autoregressive Inference for affi
==========================================================================
Key improvements over the GRU baseline:
  1. Transformer encoder-decoder (multi-head attention captures long-range deps)
  2. Autoregressive rollout at inference (each step uses the PREVIOUS PREDICTION,
     not a dummy repeated token — this is the biggest single fix)
  3. Teacher-forcing during training with scheduled sampling
  4. Cosine-annealing LR with warm-up
  5. Gradient clipping
  6. Per-channel input projection + output de-projection
  7. Optional test-time ensembling (autoregressive + direct)
"""

import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG  (change dataset_name / paths as needed)
# ──────────────────────────────────────────────────────────────────────────────
input_dir    = './'
dataset_name = 'affi'          # 'affi' or 'beignet'
num_channels = 239 if dataset_name == 'affi' else 89

BATCH_SIZE   = 32
NUM_EPOCHS   = 300
LR           = 3e-4
WARMUP_STEPS = 500            # LR warm-up in optimizer steps
D_MODEL      = 256            # transformer hidden dim
N_HEADS      = 8
N_LAYERS     = 4              # encoder layers (decoder shares same depth)
FFN_DIM      = 512
DROPOUT      = 0.1
INIT_STEPS   = 10             # conditioning window
TEACHER_FORCING_RATIO = 0.5   # 50 % teacher forcing; decays to 0 over training
GRAD_CLIP    = 1.0
SAVE_BEST    = True
CKPT_PATH    = f'model_best_{dataset_name}.pth'


# ──────────────────────────────────────────────────────────────────────────────
# DATA LOADING & NORMALISATION  (same API as baseline)
# ──────────────────────────────────────────────────────────────────────────────

def load_dataset(filename):
    test_file = os.path.join(input_dir, filename)
    data = np.load(test_file)['arr_0']          # N*T*C*F
    n = len(data)
    train_data = data[:int(n * 0.8)]
    test_data  = data[int(n * 0.8):int(n * 0.9)]
    val_data   = data[int(n * 0.9):]
    return train_data, test_data, val_data


def normalize(data, average=None, std=None):
    """Clip-normalise to [-1, 1] using mean ± 4σ."""
    if data.ndim == 4:
        n, t, c, f = data.shape
        flat = data.reshape(n * t, -1)
    else:
        flat = data
        n, t, c, f = None, None, None, None

    if average is None:
        average = np.mean(flat, axis=0, keepdims=True)
        std     = np.std(flat,  axis=0, keepdims=True)

    lo  = average - 4 * std
    hi  = average + 4 * std
    out = 2 * (flat - lo) / (hi - lo + 1e-8) - 1

    if n is not None:
        out = out.reshape(n, t, c, f)
    return out, average, std


class NeuroForcastDataset(Dataset):
    def __init__(self, neural_data, use_graph=False, average=None, std=None):
        self.use_graph = use_graph
        if average is None:
            self.data, self.average, self.std = normalize(neural_data)
        else:
            self.data, self.average, self.std = normalize(neural_data, average, std)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]              # T * C * F
        if not self.use_graph:
            d = d[:, :, 0]             # T * C
        return torch.tensor(d, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────────────────────
# POSITIONAL ENCODING
# ──────────────────────────────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # 1 * T * D

    def forward(self, x):                             # x: B * T * D
        return self.dropout(x + self.pe[:, :x.size(1)])


# ──────────────────────────────────────────────────────────────────────────────
# TRANSFORMER ENCODER-DECODER MODEL
# ──────────────────────────────────────────────────────────────────────────────

class NFTransformer(nn.Module):
    """
    Input  : B * T * C  (T time steps, C channels/neurons)
    Output : B * T * C

    Architecture
    ────────────
    • Linear projection  C → D_MODEL
    • Sinusoidal positional encoding
    • N-layer Transformer encoder (attends over time)
    • N-layer Transformer decoder (cross-attends encoder memory)
    • Linear projection  D_MODEL → C
    """
    def __init__(self, input_size, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4,
                 ffn_dim=512, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model

        self.input_proj  = nn.Linear(input_size, d_model)
        self.output_proj = nn.Linear(d_model, input_size)

        self.enc_pe = SinusoidalPE(d_model, max_len, dropout)
        self.dec_pe = SinusoidalPE(d_model, max_len, dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src):
        """src: B * T_src * C  →  memory: B * T_src * D"""
        x = self.enc_pe(self.input_proj(src))
        return self.encoder(x)

    def decode_step(self, tgt, memory):
        """tgt: B * T_tgt * C  →  out: B * T_tgt * C"""
        x = self.dec_pe(self.input_proj(tgt))
        # causal mask so decoder can't peek at future tokens
        T = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=tgt.device)
        out = self.decoder(x, memory, tgt_mask=causal)
        return self.output_proj(out)

    def forward(self, src, tgt):
        """
        Training forward pass (teacher-forced).
        src : B * T_src * C   — conditioning context (INIT_STEPS)
        tgt : B * T_tgt * C   — target sequence (teacher-forced input, shifted)
        returns: B * T_tgt * C
        """
        memory = self.encode(src)
        return self.decode_step(tgt, memory)

    @torch.no_grad()
    def autoregressive_predict(self, src, future_steps):
        """
        Inference: roll out one step at a time.
        src         : B * T_src * C
        future_steps: int — how many new steps to generate
        returns     : B * future_steps * C
        """
        memory   = self.encode(src)
        # start token = last conditioning step
        token    = src[:, -1:, :]          # B * 1 * C
        preds    = []
        tgt_buf  = token

        for _ in range(future_steps):
            out   = self.decode_step(tgt_buf, memory)   # B * t * C
            nxt   = out[:, -1:, :]                      # B * 1 * C  (last step)
            preds.append(nxt)
            tgt_buf = torch.cat([tgt_buf, nxt], dim=1)

        return torch.cat(preds, dim=1)                   # B * future_steps * C


# ──────────────────────────────────────────────────────────────────────────────
# TRAINER
# ──────────────────────────────────────────────────────────────────────────────

class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader,
                 optimizer, scheduler, device, init_steps=10,
                 save_path=CKPT_PATH, teacher_forcing_ratio=0.5):
        self.model  = model.to(device)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.test_loader  = test_loader
        self.optimizer    = optimizer
        self.scheduler    = scheduler
        self.device       = device
        self.init_steps   = init_steps
        self.save_path    = save_path
        self.tf_ratio     = teacher_forcing_ratio
        self.loss_fn      = nn.MSELoss()
        self.best_val     = float('inf')
        self.step         = 0

    # ── helpers ──────────────────────────────────────────────────────────────

    def _split(self, batch):
        """Returns conditioning context (src) and full sequence for loss (tgt_full)."""
        src      = batch[:, :self.init_steps, :]           # B * 10 * C
        tgt_full = batch[:, self.init_steps:, :]           # B * rest * C
        return src, tgt_full

    # ── train one epoch ──────────────────────────────────────────────────────

    def train_epoch(self, epoch, num_epochs):
        self.model.train()
        total_loss = 0.0
        # decay teacher-forcing linearly to 0 over training
        tf = self.tf_ratio * (1 - epoch / num_epochs)

        for batch in self.train_loader:
            batch = batch.to(self.device)
            src, tgt_future = self._split(batch)      # src: B*10*C, future: B*T'*C

            # ── build decoder input with scheduled sampling ──────────────────
            # teacher-forced target: [last conditioning step] + future[:-1]
            dec_in_tf = torch.cat([src[:, -1:, :], tgt_future[:, :-1, :]], dim=1)

            if tf > 0 and torch.rand(1).item() < tf:
                dec_in = dec_in_tf
            else:
                # use previous *prediction* as input (greedy scheduled sampling)
                with torch.no_grad():
                    pred_ss = self.model.autoregressive_predict(src, tgt_future.size(1))
                dec_in = torch.cat([src[:, -1:, :], pred_ss[:, :-1, :].detach()], dim=1)

            memory  = self.model.encode(src)
            output  = self.model.decode_step(dec_in, memory)   # B * T' * C
            loss    = self.loss_fn(output, tgt_future)

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
            self.optimizer.step()
            self.scheduler.step()
            self.step += 1
            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    # ── validation ───────────────────────────────────────────────────────────

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        total_loss = 0.0
        for batch in loader:
            batch = batch.to(self.device)
            src, tgt_future = self._split(batch)
            pred  = self.model.autoregressive_predict(src, tgt_future.size(1))
            total_loss += self.loss_fn(pred, tgt_future).item()
        return total_loss / len(loader)

    # ── full training loop ───────────────────────────────────────────────────

    def train(self, num_epochs):
        for epoch in range(1, num_epochs + 1):
            t0         = time.time()
            train_loss = self.train_epoch(epoch, num_epochs)

            if epoch % 5 == 0 or epoch == 1:
                val_loss = self.validate(self.val_loader)
                print(f"[Epoch {epoch:4d}/{num_epochs}] "
                      f"train={train_loss:.5f}  val={val_loss:.5f}  "
                      f"lr={self.scheduler.get_last_lr()[0]:.2e}  "
                      f"time={time.time()-t0:.1f}s")

                if SAVE_BEST and val_loss < self.best_val:
                    self.best_val = val_loss
                    torch.save(self.model.state_dict(), self.save_path)
                    print(f"  ↳ Best model saved  (val={val_loss:.5f})")

    # ── inference ────────────────────────────────────────────────────────────

    @torch.no_grad()
    def predict(self, loader):
        self.model.eval()
        preds, gts = [], []
        for batch in loader:
            batch = batch.to(self.device)
            src, tgt_future = self._split(batch)
            pred = self.model.autoregressive_predict(src, tgt_future.size(1))
            preds.append(pred.cpu().numpy())
            gts.append(tgt_future.cpu().numpy())
        return np.concatenate(preds, 0), np.concatenate(gts, 0)


# ──────────────────────────────────────────────────────────────────────────────
# WARM-UP + COSINE ANNEALING SCHEDULER
# ──────────────────────────────────────────────────────────────────────────────

def get_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ──────────────────────────────────────────────────────────────────────────────
# METRICS
# ──────────────────────────────────────────────────────────────────────────────

def mse_np(gt, pred):  return np.mean((gt - pred) ** 2)
def r2_np(gt, pred):   return 1 - np.sum((gt - pred) ** 2) / np.sum(gt ** 2)


# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # ── data ─────────────────────────────────────────────────────────────────
    train_raw, test_raw, val_raw = load_dataset(f'train_data_{dataset_name}.npz')
    print(f"Shapes — train:{train_raw.shape}  test:{test_raw.shape}  val:{val_raw.shape}")

    train_ds = NeuroForcastDataset(train_raw, use_graph=False)
    np.savez(f'norm_stats_{dataset_name}.npz',
             average=train_ds.average, std=train_ds.std)

    val_ds  = NeuroForcastDataset(val_raw,  use_graph=False,
                                  average=train_ds.average, std=train_ds.std)
    test_ds = NeuroForcastDataset(test_raw, use_graph=False,
                                  average=train_ds.average, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    # ── model ─────────────────────────────────────────────────────────────────
    model = NFTransformer(
        input_size        = num_channels,
        d_model           = D_MODEL,
        nhead             = N_HEADS,
        num_encoder_layers= N_LAYERS,
        num_decoder_layers= N_LAYERS,
        ffn_dim           = FFN_DIM,
        dropout           = DROPOUT,
    )
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {total_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    total_steps = NUM_EPOCHS * len(train_loader)
    scheduler   = get_scheduler(optimizer, WARMUP_STEPS, total_steps)

    trainer = Trainer(
        model, train_loader, val_loader, test_loader,
        optimizer, scheduler, device,
        init_steps          = INIT_STEPS,
        save_path           = CKPT_PATH,
        teacher_forcing_ratio = TEACHER_FORCING_RATIO,
    )

    # ── train ─────────────────────────────────────────────────────────────────
    trainer.train(NUM_EPOCHS)

    # ── evaluate on test set using best checkpoint ────────────────────────────
    print("\nLoading best checkpoint for test evaluation …")
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    model.to(device)

    preds, gts = trainer.predict(test_loader)
    print(f"\n[Test Results]")
    print(f"  MSE : {mse_np(gts, preds):.4f}")
    print(f"  R²  : {r2_np(gts, preds):.4f}")

    # ── save final predictions (for submission if needed) ─────────────────────
    np.savez(f'test_predictions_{dataset_name}.npz', preds=preds, gts=gts)
    print("Predictions saved.")


if __name__ == '__main__':
    main()

Device: cpu
Shapes — train:(788, 20, 239, 9)  test:(98, 20, 239, 9)  val:(99, 20, 239, 9)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Parameters: 5,394,415


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Epoch    1/300] train=65.25294  val=20.06006  lr=1.50e-05  time=21.7s
  ↳ Best model saved  (val=20.06006)
[Epoch    5/300] train=3.47615  val=0.93295  lr=7.50e-05  time=19.7s
  ↳ Best model saved  (val=0.93295)
[Epoch   10/300] train=0.95490  val=0.18795  lr=1.50e-04  time=26.1s
  ↳ Best model saved  (val=0.18795)
[Epoch   15/300] train=0.26573  val=0.05542  lr=2.25e-04  time=23.8s
  ↳ Best model saved  (val=0.05542)
[Epoch   20/300] train=0.10101  val=0.02792  lr=3.00e-04  time=22.2s
  ↳ Best model saved  (val=0.02792)
[Epoch   25/300] train=0.06130  val=0.02506  lr=3.00e-04  time=22.4s
  ↳ Best model saved  (val=0.02506)
[Epoch   30/300] train=0.04801  val=0.01928  lr=2.99e-04  time=23.9s
  ↳ Best model saved  (val=0.01928)
[Epoch   35/300] train=0.03995  val=0.02051  lr=2.98e-04  time=24.6s
[Epoch   40/300] train=0.03371  val=0.01929  lr=2.96e-04  time=23.6s
[Epoch   45/300] train=0.02852  val=0.01676  lr=2.94e-04  time=24.5s
  ↳ Best model saved  (val=0.01676)
[Epoch   50/300] tr

for beignet

In [ ]:
"""
NeuroForcast — Transformer Encoder-Decoder with Autoregressive Inference for Beignet
==========================================================================
Key improvements over the GRU baseline:
  1. Transformer encoder-decoder (multi-head attention captures long-range deps)
  2. Autoregressive rollout at inference (each step uses the PREVIOUS PREDICTION,
     not a dummy repeated token — this is the biggest single fix)
  3. Teacher-forcing during training with scheduled sampling
  4. Cosine-annealing LR with warm-up
  5. Gradient clipping
  6. Per-channel input projection + output de-projection
  7. Optional test-time ensembling (autoregressive + direct)
"""

import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG  (change dataset_name / paths as needed)
# ──────────────────────────────────────────────────────────────────────────────
input_dir    = './'
dataset_name = 'beignet'          # 'affi' or 'beignet'
num_channels = 239 if dataset_name == 'affi' else 89

BATCH_SIZE   = 32
NUM_EPOCHS   = 300
LR           = 3e-4
WARMUP_STEPS = 500            # LR warm-up in optimizer steps
D_MODEL      = 256            # transformer hidden dim
N_HEADS      = 8
N_LAYERS     = 4              # encoder layers (decoder shares same depth)
FFN_DIM      = 512
DROPOUT      = 0.1
INIT_STEPS   = 10             # conditioning window
TEACHER_FORCING_RATIO = 0.5   # 50 % teacher forcing; decays to 0 over training
GRAD_CLIP    = 1.0
SAVE_BEST    = True
CKPT_PATH    = f'model_best_{dataset_name}.pth'


# ──────────────────────────────────────────────────────────────────────────────
# DATA LOADING & NORMALISATION  (same API as baseline)
# ──────────────────────────────────────────────────────────────────────────────

def load_dataset(filename):
    test_file = os.path.join(input_dir, filename)
    data = np.load(test_file)['arr_0']          # N*T*C*F
    n = len(data)
    train_data = data[:int(n * 0.8)]
    test_data  = data[int(n * 0.8):int(n * 0.9)]
    val_data   = data[int(n * 0.9):]
    return train_data, test_data, val_data


def normalize(data, average=None, std=None):
    """Clip-normalise to [-1, 1] using mean ± 4σ."""
    if data.ndim == 4:
        n, t, c, f = data.shape
        flat = data.reshape(n * t, -1)
    else:
        flat = data
        n, t, c, f = None, None, None, None

    if average is None:
        average = np.mean(flat, axis=0, keepdims=True)
        std     = np.std(flat,  axis=0, keepdims=True)

    lo  = average - 4 * std
    hi  = average + 4 * std
    out = 2 * (flat - lo) / (hi - lo + 1e-8) - 1

    if n is not None:
        out = out.reshape(n, t, c, f)
    return out, average, std


class NeuroForcastDataset(Dataset):
    def __init__(self, neural_data, use_graph=False, average=None, std=None):
        self.use_graph = use_graph
        if average is None:
            self.data, self.average, self.std = normalize(neural_data)
        else:
            self.data, self.average, self.std = normalize(neural_data, average, std)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]              # T * C * F
        if not self.use_graph:
            d = d[:, :, 0]             # T * C
        return torch.tensor(d, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────────────────────
# POSITIONAL ENCODING
# ──────────────────────────────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # 1 * T * D

    def forward(self, x):                             # x: B * T * D
        return self.dropout(x + self.pe[:, :x.size(1)])


# ──────────────────────────────────────────────────────────────────────────────
# TRANSFORMER ENCODER-DECODER MODEL
# ──────────────────────────────────────────────────────────────────────────────

class NFTransformer(nn.Module):
    """
    Input  : B * T * C  (T time steps, C channels/neurons)
    Output : B * T * C

    Architecture
    ────────────
    • Linear projection  C → D_MODEL
    • Sinusoidal positional encoding
    • N-layer Transformer encoder (attends over time)
    • N-layer Transformer decoder (cross-attends encoder memory)
    • Linear projection  D_MODEL → C
    """
    def __init__(self, input_size, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4,
                 ffn_dim=512, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model

        self.input_proj  = nn.Linear(input_size, d_model)
        self.output_proj = nn.Linear(d_model, input_size)

        self.enc_pe = SinusoidalPE(d_model, max_len, dropout)
        self.dec_pe = SinusoidalPE(d_model, max_len, dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src):
        """src: B * T_src * C  →  memory: B * T_src * D"""
        x = self.enc_pe(self.input_proj(src))
        return self.encoder(x)

    def decode_step(self, tgt, memory):
        """tgt: B * T_tgt * C  →  out: B * T_tgt * C"""
        x = self.dec_pe(self.input_proj(tgt))
        # causal mask so decoder can't peek at future tokens
        T = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=tgt.device)
        out = self.decoder(x, memory, tgt_mask=causal)
        return self.output_proj(out)

    def forward(self, src, tgt):
        """
        Training forward pass (teacher-forced).
        src : B * T_src * C   — conditioning context (INIT_STEPS)
        tgt : B * T_tgt * C   — target sequence (teacher-forced input, shifted)
        returns: B * T_tgt * C
        """
        memory = self.encode(src)
        return self.decode_step(tgt, memory)

    @torch.no_grad()
    def autoregressive_predict(self, src, future_steps):
        """
        Inference: roll out one step at a time.
        src         : B * T_src * C
        future_steps: int — how many new steps to generate
        returns     : B * future_steps * C
        """
        memory   = self.encode(src)
        # start token = last conditioning step
        token    = src[:, -1:, :]          # B * 1 * C
        preds    = []
        tgt_buf  = token

        for _ in range(future_steps):
            out   = self.decode_step(tgt_buf, memory)   # B * t * C
            nxt   = out[:, -1:, :]                      # B * 1 * C  (last step)
            preds.append(nxt)
            tgt_buf = torch.cat([tgt_buf, nxt], dim=1)

        return torch.cat(preds, dim=1)                   # B * future_steps * C


# ──────────────────────────────────────────────────────────────────────────────
# TRAINER
# ──────────────────────────────────────────────────────────────────────────────

class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader,
                 optimizer, scheduler, device, init_steps=10,
                 save_path=CKPT_PATH, teacher_forcing_ratio=0.5):
        self.model  = model.to(device)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.test_loader  = test_loader
        self.optimizer    = optimizer
        self.scheduler    = scheduler
        self.device       = device
        self.init_steps   = init_steps
        self.save_path    = save_path
        self.tf_ratio     = teacher_forcing_ratio
        self.loss_fn      = nn.MSELoss()
        self.best_val     = float('inf')
        self.step         = 0

    # ── helpers ──────────────────────────────────────────────────────────────

    def _split(self, batch):
        """Returns conditioning context (src) and full sequence for loss (tgt_full)."""
        src      = batch[:, :self.init_steps, :]           # B * 10 * C
        tgt_full = batch[:, self.init_steps:, :]           # B * rest * C
        return src, tgt_full

    # ── train one epoch ──────────────────────────────────────────────────────

    def train_epoch(self, epoch, num_epochs):
        self.model.train()
        total_loss = 0.0
        # decay teacher-forcing linearly to 0 over training
        tf = self.tf_ratio * (1 - epoch / num_epochs)

        for batch in self.train_loader:
            batch = batch.to(self.device)
            src, tgt_future = self._split(batch)      # src: B*10*C, future: B*T'*C

            # ── build decoder input with scheduled sampling ──────────────────
            # teacher-forced target: [last conditioning step] + future[:-1]
            dec_in_tf = torch.cat([src[:, -1:, :], tgt_future[:, :-1, :]], dim=1)

            if tf > 0 and torch.rand(1).item() < tf:
                dec_in = dec_in_tf
            else:
                # use previous *prediction* as input (greedy scheduled sampling)
                with torch.no_grad():
                    pred_ss = self.model.autoregressive_predict(src, tgt_future.size(1))
                dec_in = torch.cat([src[:, -1:, :], pred_ss[:, :-1, :].detach()], dim=1)

            memory  = self.model.encode(src)
            output  = self.model.decode_step(dec_in, memory)   # B * T' * C
            loss    = self.loss_fn(output, tgt_future)

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
            self.optimizer.step()
            self.scheduler.step()
            self.step += 1
            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    # ── validation ───────────────────────────────────────────────────────────

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        total_loss = 0.0
        for batch in loader:
            batch = batch.to(self.device)
            src, tgt_future = self._split(batch)
            pred  = self.model.autoregressive_predict(src, tgt_future.size(1))
            total_loss += self.loss_fn(pred, tgt_future).item()
        return total_loss / len(loader)

    # ── full training loop ───────────────────────────────────────────────────

    def train(self, num_epochs):
        for epoch in range(1, num_epochs + 1):
            t0         = time.time()
            train_loss = self.train_epoch(epoch, num_epochs)

            if epoch % 5 == 0 or epoch == 1:
                val_loss = self.validate(self.val_loader)
                print(f"[Epoch {epoch:4d}/{num_epochs}] "
                      f"train={train_loss:.5f}  val={val_loss:.5f}  "
                      f"lr={self.scheduler.get_last_lr()[0]:.2e}  "
                      f"time={time.time()-t0:.1f}s")

                if SAVE_BEST and val_loss < self.best_val:
                    self.best_val = val_loss
                    torch.save(self.model.state_dict(), self.save_path)
                    print(f"  ↳ Best model saved  (val={val_loss:.5f})")

    # ── inference ────────────────────────────────────────────────────────────

    @torch.no_grad()
    def predict(self, loader):
        self.model.eval()
        preds, gts = [], []
        for batch in loader:
            batch = batch.to(self.device)
            src, tgt_future = self._split(batch)
            pred = self.model.autoregressive_predict(src, tgt_future.size(1))
            preds.append(pred.cpu().numpy())
            gts.append(tgt_future.cpu().numpy())
        return np.concatenate(preds, 0), np.concatenate(gts, 0)


# ──────────────────────────────────────────────────────────────────────────────
# WARM-UP + COSINE ANNEALING SCHEDULER
# ──────────────────────────────────────────────────────────────────────────────

def get_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ──────────────────────────────────────────────────────────────────────────────
# METRICS
# ──────────────────────────────────────────────────────────────────────────────

def mse_np(gt, pred):  return np.mean((gt - pred) ** 2)
def r2_np(gt, pred):   return 1 - np.sum((gt - pred) ** 2) / np.sum(gt ** 2)


# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # ── data ─────────────────────────────────────────────────────────────────
    train_raw, test_raw, val_raw = load_dataset(f'train_data_{dataset_name}.npz')
    print(f"Shapes — train:{train_raw.shape}  test:{test_raw.shape}  val:{val_raw.shape}")

    train_ds = NeuroForcastDataset(train_raw, use_graph=False)
    np.savez(f'norm_stats_{dataset_name}.npz',
             average=train_ds.average, std=train_ds.std)

    val_ds  = NeuroForcastDataset(val_raw,  use_graph=False,
                                  average=train_ds.average, std=train_ds.std)
    test_ds = NeuroForcastDataset(test_raw, use_graph=False,
                                  average=train_ds.average, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    # ── model ─────────────────────────────────────────────────────────────────
    model = NFTransformer(
        input_size        = num_channels,
        d_model           = D_MODEL,
        nhead             = N_HEADS,
        num_encoder_layers= N_LAYERS,
        num_decoder_layers= N_LAYERS,
        ffn_dim           = FFN_DIM,
        dropout           = DROPOUT,
    )
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {total_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    total_steps = NUM_EPOCHS * len(train_loader)
    scheduler   = get_scheduler(optimizer, WARMUP_STEPS, total_steps)

    trainer = Trainer(
        model, train_loader, val_loader, test_loader,
        optimizer, scheduler, device,
        init_steps          = INIT_STEPS,
        save_path           = CKPT_PATH,
        teacher_forcing_ratio = TEACHER_FORCING_RATIO,
    )

    # ── train ─────────────────────────────────────────────────────────────────
    trainer.train(NUM_EPOCHS)

    # ── evaluate on test set using best checkpoint ────────────────────────────
    print("\nLoading best checkpoint for test evaluation …")
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    model.to(device)

    preds, gts = trainer.predict(test_loader)
    print(f"\n[Test Results]")
    print(f"  MSE : {mse_np(gts, preds):.4f}")
    print(f"  R²  : {r2_np(gts, preds):.4f}")

    # ── save final predictions (for submission if needed) ─────────────────────
    np.savez(f'test_predictions_{dataset_name}.npz', preds=preds, gts=gts)
    print("Predictions saved.")


if __name__ == '__main__':
    main()

Device: cpu
Shapes — train:(560, 20, 89, 9)  test:(70, 20, 89, 9)  val:(70, 20, 89, 9)


/tmp/ipython-input-2046955452.py:151: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)


Parameters: 5,317,465


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Epoch    1/300] train=40.19161  val=22.82406  lr=1.08e-05  time=12.6s
  ↳ Best model saved  (val=22.82406)
[Epoch    5/300] train=5.99603  val=1.87430  lr=5.40e-05  time=12.7s
  ↳ Best model saved  (val=1.87430)
[Epoch   10/300] train=1.93643  val=0.48067  lr=1.08e-04  time=14.3s
  ↳ Best model saved  (val=0.48067)
[Epoch   15/300] train=0.70763  val=0.17777  lr=1.62e-04  time=11.5s
  ↳ Best model saved  (val=0.17777)
[Epoch   20/300] train=0.36905  val=0.08705  lr=2.16e-04  time=13.5s
  ↳ Best model saved  (val=0.08705)
[Epoch   25/300] train=0.20710  val=0.05940  lr=2.70e-04  time=15.0s
  ↳ Best model saved  (val=0.05940)
[Epoch   30/300] train=0.13146  val=0.05892  lr=3.00e-04  time=15.2s
  ↳ Best model saved  (val=0.05892)
[Epoch   35/300] train=0.09385  val=0.04805  lr=2.99e-04  time=13.6s
  ↳ Best model saved  (val=0.04805)
[Epoch   40/300] train=0.07700  val=0.04358  lr=2.99e-04  time=14.5s
  ↳ Best model saved  (val=0.04358)
[Epoch   45/300] train=0.06394  val=0.04004  lr=2.97

KeyboardInterrupt: 